In [2]:
import dash
import dash_table
from dash import html, dcc, Input, Output, callback
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from dash.exceptions import PreventUpdate

#Importation base et traitement#
#base 1
Incidence = pd.read_csv("./ILINet.csv",header=1)
Incidence = Incidence[['REGION', 'YEAR','WEEK','AGE 0-4','AGE 25-49','AGE 5-24',
                      'AGE 50-64','AGE 65','ILITOTAL']]

#base 2
Catégorie = pd.read_csv("./WHO_NREVSS_Clinical_Labs.csv",header=1)
Catégorie = Catégorie[['REGION', 'YEAR','WEEK','TOTAL SPECIMENS','TOTAL A','TOTAL B']]

#Fusion des data frame
df = pd.merge(Incidence, Catégorie, on=['YEAR','WEEK','REGION'], how='right')

#Graphique 1 - Evolution de l'incidence de la grippe
def plot_incidence_evolution(data):
    data['DATE'] = data.apply(lambda x: pd.Timestamp(year=x['YEAR'], month=1, day=1) + pd.Timedelta(weeks=x['WEEK']-1), axis=1)
    data_grouped = data.groupby('DATE')['ILITOTAL'].sum().reset_index()
    fig = px.line(data_grouped, x='DATE', y='ILITOTAL', title='Évolution de l\'incidence des symptômes grippaux par semaine')
    fig.update_xaxes(title='Semaine')
    fig.update_yaxes(title='Incidence des symptômes grippaux')
    fig.update_traces(line=dict(color='#9DC183'))
    fig.update_layout(template='plotly', plot_bgcolor='#F4F4F4')
    return fig   

#Données annuel
def Données_annuel(data):
    return data.groupby('YEAR').sum()

#Camenbert 1 - Catégorie de virus par ans
def create_pie_chart_from_database(data):
    df = Données_annuel(data)
    total_specimens = df['TOTAL SPECIMENS'].sum()
    total_a = df['TOTAL A'].sum()
    total_b = df['TOTAL B'].sum()
    total_negatif = total_specimens - total_a - total_b
    data_values = [total_a, total_b, total_negatif]
    labels = ['Grippe A', 'Grippe B', 'Négatif']
    colors = ['#FADADD', '#B0E0E6', '#9DC183']
    fig = go.Figure(data=[go.Pie(labels=labels, values=data_values, hole=0.4)])
    fig.update_traces(hoverinfo='label+percent', textinfo='value', textfont_size=10,
                      marker=dict(colors=colors, line=dict(color='#000000', width=2)))
    fig.update_layout(title='Répartition & Résultat des tests', legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
    return fig


#Histogramme 1 - répartition par catégorie d'âges
def plot_age_incidence(data):
    sum_values = data[['AGE 0-4', 'AGE 5-24', 'AGE 25-49', 'AGE 50-64', 'AGE 65']].sum()
    colors = ['#FADADD', '#B0E0E6', '#9DC183', '#E6BE8A', '#E6E6FA']
    bars = go.Bar(x=sum_values.index, y=sum_values.values, marker_color=colors)
    fig = go.Figure(bars)
    fig.update_layout(
        title="Incidence des symptômes grippaux par catégorie d'âge",
        xaxis=dict(title="Catégorie d'âge"),
        yaxis=dict(title="Incidence des symptômes grippaux"),
        plot_bgcolor='#f4f4f4' 
    )
    fig.update_layout(legend=dict(
        x=1,
        y=1,
        traceorder='normal',
        font=dict(size=10,),
    ))
    return fig
    
#Carte 1 - Incidence par Census division
def plot_incidence_map(data):
    def Données_region(data):
        return data.groupby(['REGION','YEAR']).sum().reset_index()
    region_coordinates = pd.DataFrame({
        'REGION': ['New England', 'Mid-Atlantic', 'East North Central', 'West North Central', 'South Atlantic', 'East South Central', 'West South Central', 'Mountain', 'Pacific'],
        'Latitude': [42.3601, 39.9526, 41.8781, 41.2565, 33.7490, 36.1627, 29.7604, 39.7392, 34.0522],
        'Longitude': [-71.0589, -75.1652, -87.6298, -95.9345, -84.3880, -86.7816, -95.3698, -104.9903, -118.2437]
    })
    incidence_data = Données_region(data)
    incidence_data = pd.merge(incidence_data, region_coordinates, on='REGION', how='right')
    # Tracer la carte des incidences
    fig = px.scatter_mapbox(
        incidence_data,
        lat="Latitude",
        lon="Longitude",
        color="ILITOTAL",
        size="ILITOTAL",
        color_continuous_scale=px.colors.cyclical.IceFire,
        size_max=50, 
        zoom=2,  
        mapbox_style="carto-positron",
        hover_name="REGION"
    )
    fig.update_layout(title="Carte de l'incidence des symptômes grippaux par région")
    return fig


#Dash 
# Dash 
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
colors = {
    'background': '#cbb1b4',
    'background2': '#f2e9ea'
}

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.Div(html.Strong("Info-Grippe"), style={"text-align": "center", "line-height": "100px", "font-size": "20px"}), width=4),
        dbc.Col([html.Span("\u200B")], width=4),
        dbc.Col([
            dbc.Row(["\u200B"], style={"background-color": colors['background']}),
            dbc.Row([
                dcc.Dropdown(
                    id='year',
                    options=[{'label': str(year), 'value': year} for year in df['YEAR'].dropna().unique()],
                    value=df['YEAR'].dropna().unique()[0],
                    clearable=False,
                    multi=False,
                    searchable=False,
                    placeholder="Sélection de l'année"
                        )
            ]),
            dbc.Row([
                dcc.Dropdown(
                    id='region',
                    options=[{'label': region, 'value': region} for region in df['REGION'].dropna().unique()],
                    multi=True,
                    searchable=True,
                    placeholder="Sélectionnez la région"
                )
            ]),
        ], width=4)
    ], style={"background-color": colors['background']}),
    dbc.Row([
        dbc.Col(children=dcc.Graph(id='Carte-1'), style={'background-color': colors['background2']}, width=6),
        dbc.Col(children=dcc.Graph(id='Graphique-2'), style={'background-color': colors['background2']}, width=6),
    ],style={"background-color": colors['background2']}),
    dbc.Row([
        dbc.Col(children=dcc.Graph(id='Graphique-1'), style={'background-color': colors['background2']}, width=6),
        dbc.Col(children=dcc.Graph(id='Camembert-1'), style={'background-color': colors['background2']},width=6),
    ],style={"background-color": colors['background2']})
], fluid=True)

@app.callback(
    Output('Graphique-1', 'figure'),
    [Input('region', 'value'), Input('year', 'value')]
)
def plot_age_incidence_2(region, year):
    if year and not region:
        df_temps = df[df['YEAR'].isin([year])]
    elif region and not year:
        df_temps = df[df['REGION'].isin(region)]
    elif year and region:
        df_temps = df[df['REGION'].isin(region) & df['YEAR'].isin([year])]
    else:
        df_temps = df
    return plot_age_incidence(df_temps)

@app.callback(
    Output('Graphique-2', 'figure'),
    [Input('region', 'value'), Input('year', 'value')]
)
def plot_incidence_evolution2(region, year):
    if year and not region:
        df_temps = df[df['YEAR'].isin([year])]
    elif region and not year:
        df_temps = df[df['REGION'].isin(region)]
    elif year and region:
        df_temps = df[df['REGION'].isin(region) & df['YEAR'].isin([year])]
    else:
        df_temps = df
    return plot_incidence_evolution(df_temps)

@app.callback(
    Output('Camembert-1', 'figure'),
    [Input('region', 'value'), Input('year', 'value')]
)
def create_pie_chart_from_database2(region, year):
    if year and not region:
        df_temps = df[df['YEAR'].isin([year])]
    elif region and not year:
        df_temps = df[df['REGION'].isin(region)]
    elif year and region:
        df_temps = df[df['REGION'].isin(region) & df['YEAR'].isin([year])]
    else:
        df_temps = df
    return create_pie_chart_from_database(df_temps)

@app.callback(
    Output('Carte-1', 'figure'),
    [Input('region', 'value'), Input('year', 'value')]
)
def plot_incidence_map2(region, year):
    if year and not region:
        df_temps = df[df['YEAR'].isin([year])]
    else:
        df_temps = df[df['YEAR'].isin([year])]
    return plot_incidence_map(df_temps)


if __name__ == '__main__':
    app.run_server(debug=True, port=8053, jupyter_mode="external")


Dash app running on http://127.0.0.1:8053/


/var/folders/jq/dwvfq6hd6cb00_ztp8scl1_h0000gn/T/ipykernel_11230/650492027.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/jq/dwvfq6hd6cb00_ztp8scl1_h0000gn/T/ipykernel_11230/650492027.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/jq/dwvfq6hd6cb00_ztp8scl1_h0000gn/T/ipykernel_11230/650492027.py:26: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d